## Bài tập chương 6 - Neural Network

### Nhóm 6
Thành viên thực hiện:
- Hà Xuân Thiện - 24520031
- Hà Thanh Phong - 24520024
- Trần Quang Trường - 24521901

Trong bài tập lần này, ta sẽ huấn luyện mô hình FeedFoward Neural Network (FNN) trên bộ dữ liệu [`VN Topic Classification Dataset
`](https://www.kaggle.com/datasets/moulsn/vn-topic-classification-dataset/).

Bộ dữ liệu này chứa khoảng 5.000 nội dung tin tức tiếng Việt, thu thập từ các nguồn truyền thông lớn của Việt Nam (ví dụ: vov.vn, vietnamnet.vn, vnexpress.net, thanhnien.vn, baovanhoa.vn). Dữ liệu đã được làm sạch và gắn nhãn.

Nhiệm vụ của ta là cần huấn luyện mô hình để dự đoán được chủ đề (topic) từ nội dung tin tức.

Import những thư viện cần thiết.

In [3]:
import re
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

Set seed để đảm bảo tính nhất quán mỗi lần chạy.

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


Đọc dữ liệu raw.

In [5]:
df_raw = pd.read_csv("train.csv")
df_raw.head()

,text,label
0,"Hội thảo khoa học về sức mạnh chính trị, tinh ...",Chính trị
1,"""Sáp nhập tỉnh dự kiến hoàn thành trước ngày 3...",Chính trị
2,Chủ tịch Quốc hội làm việc với Ban Thường vụ T...,Chính trị
3,Thủ tướng chủ trì Phiên họp thứ nhất BCĐ của C...,Chính trị
4,Ông Nguyễn Mạnh Hùng giữ chức Bí thư Tỉnh ủy T...,Chính trị


Tiền xử lý dữ liệu.

In [6]:
def clean_text(x):
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df = df_raw.copy()
df["text"] = df["text"].fillna("").apply(clean_text)
df["label"] = df["label"].astype(str).str.strip()

print(df.isna().sum())

text     0
label    0
dtype: int64


Encoder nhãn dự đoán.

In [7]:
label_encoder = LabelEncoder()
X = df["text"].values
y = label_encoder.fit_transform(df["label"])

print("Classes:", list(label_encoder.classes_))

Classes: ['Chính trị', 'Công nghệ', 'Kinh tế', 'Thể thao', 'văn hoá']


Chia tập dữ liệu thành train, dev, test.

In [8]:
X_train, X_dev_test, y_train, y_dev_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_dev_test, y_dev_test,
    test_size=0.5,
    random_state=SEED,
    stratify=y_dev_test
)

print("Train size:", len(X_train))
print("Dev size:", len(X_dev))
print("Test size:", len(X_test))

Train size: 3996
Dev size: 499
Test size: 500


Biểu diễn văn bản bằng TF‑IDF.

In [9]:
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train)
X_dev   = vectorizer.transform(X_dev)
X_test  = vectorizer.transform(X_test)

Tạo dataloader cho tập train, dev, test.

In [10]:
class TextClassificationDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = torch.as_tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        feature_vector = self.features[index]
        feature_vector = feature_vector.toarray()[0]
        feature_vector = torch.tensor(feature_vector, dtype=torch.float32)

        label = self.labels[index]
        return feature_vector, label


train_dataset = TextClassificationDataset(X_train, y_train)
dev_dataset   = TextClassificationDataset(X_dev, y_dev)
test_dataset  = TextClassificationDataset(X_test, y_test)

batch_size = 64 

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dataset=dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

Tạo mô hình huấn luận.

In [11]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = FeedForwardNN(
    input_dim=X_train.shape[1],
    num_classes=len(label_encoder.classes_)
).to(device)

print(model)

FeedForwardNN(
  (net): Sequential(
    (0): Linear(in_features=15000, out_features=512, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=128, out_features=5, bias=True)
  )
)


Thiết lập criterion và optimizer. Ở đây, ta chọn hàm loss Cross Entropy, và optimizer sử dụng là AdamW. 

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

Ta tạo một hàm dùng chung cho cả tập train, dev, test cho từng epoch.  
Nếu chuyền vào tập train, thì hàm sẽ chạy với ý nghĩa là huấn luận mô hình. 
Còn nếu đưa vào tập dev với test thì hàm sẽ chạy với ý nghĩa tìm acc, f1_score.

In [13]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for texts, labels in loader:
        texts = texts.to(device)
        labels = labels.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(texts) 
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * texts.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1, all_labels, all_preds

Huấn luận mô hình.

In [14]:
num_epochs = 36
patience = 3

best_val_f1 = -1
best_state = None
wait = 0

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc, train_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, _, _ = run_epoch(model, dev_loader, criterion, optimizer=None)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

model.load_state_dict(best_state)
print("Best val macro-F1:", best_val_f1)

Epoch 01 | train_loss=0.1738 train_acc=0.9502 train_f1=0.9502 | val_loss=0.0950 val_acc=0.9840 val_f1=0.9838
Epoch 02 | train_loss=0.0175 train_acc=0.9957 train_f1=0.9957 | val_loss=0.0606 val_acc=0.9880 val_f1=0.9879
Epoch 03 | train_loss=0.0119 train_acc=0.9977 train_f1=0.9977 | val_loss=0.0776 val_acc=0.9840 val_f1=0.9839
Epoch 04 | train_loss=0.0083 train_acc=0.9977 train_f1=0.9977 | val_loss=0.0663 val_acc=0.9880 val_f1=0.9880
Epoch 05 | train_loss=0.0052 train_acc=0.9985 train_f1=0.9985 | val_loss=0.0704 val_acc=0.9860 val_f1=0.9859
Epoch 06 | train_loss=0.0122 train_acc=0.9975 train_f1=0.9975 | val_loss=0.1012 val_acc=0.9739 val_f1=0.9736
Epoch 07 | train_loss=0.0064 train_acc=0.9990 train_f1=0.9990 | val_loss=0.0952 val_acc=0.9780 val_f1=0.9778
Early stopping.
Best val macro-F1: 0.9879580198233991


Kiểm tra kết quả trên tập test.

In [15]:
test_loss, test_acc, test_f1, y_true, y_pred = run_epoch(model, test_loader, criterion, optimizer=None)

print("\nTest loss:", round(test_loss, 4))
print("Test acc :", round(test_acc, 4))
print("Test F1  :", round(test_f1, 4))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4
))


Test loss: 0.0677
Test acc : 0.976
Test F1  : 0.9759

Classification report:
              precision    recall  f1-score   support

   Chính trị     1.0000    0.9900    0.9950       100
   Công nghệ     0.9709    1.0000    0.9852       100
     Kinh tế     0.9894    0.9300    0.9588       100
    Thể thao     0.9804    1.0000    0.9901       100
     văn hoá     0.9412    0.9600    0.9505       100

    accuracy                         0.9760       500
   macro avg     0.9764    0.9760    0.9759       500
weighted avg     0.9764    0.9760    0.9759       500



Ví dụ mẫu một nội dung là "Giá xăng tăng mạnh trong một tháng vừa qua làm ảnh hưởng đến chi phí đi lại của người dân".

/// T_T huhu em cũng đi xe máy.


In [18]:
def predict_topic(text):
    text = clean_text(text)
    x = vectorizer.transform([text]).toarray().astype(np.float32)
    x = torch.tensor(x, device=device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        pred_id = torch.argmax(logits, dim=1).item()

    return label_encoder.inverse_transform([pred_id])[0]

sample_text = "Giá xăng tăng mạnh trong một tháng vừa qua làm ảnh hưởng đến chi phí đi lại của người dân."
print(predict_topic(sample_text))

Kinh tế
